# Study 897 — CPPI Floor 🧱

**Build a trapdoor-proof floor by rule alone: hold a fixed multiple of your "cushion" in stocks, the
rest in bills, and de-risk automatically as the cushion shrinks. Does the floor really hold — and what
does it cost?**

**Constant-Proportion Portfolio Insurance** (Black & Jones 1987; Perold & Sharpe 1988) promises a
mechanical downside floor with no options: set a protected floor `F` (here **80%** of NAV,
accreting at the cash rate), and each day hold `m ×` the cushion `(V − F)` in SPY — multiplier
**m = 5** — and the rest in BIL bills. As the market falls the cushion shrinks and the rule
sells stock; if it hits zero you are cash-locked on the floor. Real tape SPY/BIL,
2007-05-31 → 2026-06-30.

*Numbers below are the frozen headline (`docs/results.md`, fingerprint `4b5a30d250b4`); the live cells run
the fast synthetic control. Short history: BIL lists 2007 — a single, GFC-anchored cycle (which is
exactly where a floor should shine).*


In [1]:
R = {'alpha_ann': -1.56,
 'avg_w': 0.38,
 'bear_breach': 0,
 'bear_prot': 0.52,
 'bear_prot_min': 0.11,
 'beta': 0.705,
 'bh_cagr': 10.7,
 'bh_dd': -55.2,
 'bh_sharpe': 0.542,
 'bh_vol': 19.78,
 'boot_hi': -0.05,
 'boot_lo': -0.691,
 'boot_point': -0.379,
 'boot_win': 1.4,
 'calm_prot': 0.005,
 'cost0': -0.379,
 'cost1': -0.394,
 'cost10': -0.554,
 'cost10_dd': -17.3,
 'cost2': -0.413,
 'cost5': -0.472,
 'cppi_cagr': 2.37,
 'cppi_dd': -20.8,
 'cppi_sharpe': 0.164,
 'cppi_vol': 7.94,
 'crash08_bh': -36.8,
 'crash08_bh_dd': -47.1,
 'crash08_c': -11.6,
 'crash08_c_dd': -11.2,
 'crash20_bh': 18.3,
 'crash20_bh_dd': -33.7,
 'crash20_c': -4.8,
 'crash20_c_dd': -16.3,
 'crash22_bh': -18.2,
 'crash22_bh_dd': -24.5,
 'crash22_c': -18.7,
 'crash22_c_dd': -20.8,
 'diff_t_nw': -2.15,
 'end': '2026-06-30',
 'era_e_bh': 0.335,
 'era_e_cppi': -0.464,
 'era_e_n': 2164,
 'era_e_t': -2.38,
 'era_e_vs': -0.799,
 'era_l_bh': 0.759,
 'era_l_cppi': 0.751,
 'era_l_n': 2637,
 'era_l_t': -0.74,
 'era_l_vs': -0.008,
 'fl70_cagr': 4.83,
 'fl70_dd': -31.9,
 'fl70_sh': 0.323,
 'fl80_cagr': 2.37,
 'fl80_dd': -20.8,
 'fl80_sh': 0.164,
 'fl90_cagr': 1.78,
 'fl90_dd': -10.6,
 'fl90_sh': 0.123,
 'floor': 80,
 'fp': '4b5a30d250b4',
 'gap15_fire': 0,
 'gap25_fire': 20,
 'gap_breach_at': 20,
 'gap_thresh': 20,
 'm3_dd': -33.5,
 'm3_sh': 0.517,
 'm3_w': 0.67,
 'm4_dd': -31.4,
 'm4_sh': 0.417,
 'm5_dd': -20.8,
 'm5_sh': 0.164,
 'm6_dd': -18.0,
 'm6_sh': -0.242,
 'm8_dd': -19.2,
 'm8_sh': -0.288,
 'mult': 5,
 'n_breach': 0,
 'n_days': 4801,
 'n_seeds': 30,
 'sharpe_vs_bh': -0.379,
 'start': '2007-05-31',
 'static_cagr': 5.27,
 'static_dd': -23.0,
 'static_sharpe': 0.542,
 'static_vol': 7.42,
 't_alpha': -1.16,
 'turnover': 2.3}

## 1. The floor really holds — that part is true

The rule is purely mechanical and it does exactly what it says on the tin: it caps the drawdown. Over 2007–2026 CPPI's worst peak-to-trough was a **third** of buy-and-hold's, and the floor was **never breached**.

In [2]:
print(f"CPPI (m={R['mult']}, floor {R['floor']}%): maxDD {R['cppi_dd']:.1f}%   "
      f"floor breaches {R['n_breach']}")
print(f"buy-and-hold SPY        : maxDD {R['bh_dd']:.1f}%")
print(f"2008 crash: buy-and-hold {R['crash08_bh']:+.1f}% (DD {R['crash08_bh_dd']:.1f}%)"
      f"  ->  CPPI {R['crash08_c']:+.1f}% (DD {R['crash08_c_dd']:.1f}%)")

CPPI (m=5, floor 80%): maxDD -20.8%   floor breaches 0
buy-and-hold SPY        : maxDD -55.2%
2008 crash: buy-and-hold -36.8% (DD -47.1%)  ->  CPPI -11.6% (DD -11.2%)


## 2. …but it is paid for in full, out of the upside

Insurance is never free. The floor is bought by holding a lot of cash, so CPPI compounds at **2.4%/yr** while buy-and-hold does **10.7%** — and its risk-adjusted return (excess-of-cash **Sharpe 0.16**) is a *third* of buy-and-hold's (**0.54**). The bootstrap is emphatic: the Sharpe shortfall is real with ~99% confidence.

In [3]:
print(f"excess-Sharpe:  CPPI {R['cppi_sharpe']:.3f}  vs  buy-and-hold {R['bh_sharpe']:.3f}")
print(f"CAGR:           CPPI {R['cppi_cagr']:.2f}%   vs  buy-and-hold {R['bh_cagr']:.2f}%")
print(f"bootstrap Sharpe gain (CPPI - BH): {R['boot_point']:+.3f}  "
      f"95% CI [{R['boot_lo']:+.3f}, {R['boot_hi']:+.3f}]  P(CPPI wins) {R['boot_win']:.1f}%")

excess-Sharpe:  CPPI 0.164  vs  buy-and-hold 0.542
CAGR:           CPPI 2.37%   vs  buy-and-hold 10.70%
bootstrap Sharpe gain (CPPI - BH): -0.379  95% CI [-0.691, -0.050]  P(CPPI wins) 1.4%


## 3. The 2020 trap — mechanical insurance whipsaws on a V

A floor tuned for a *crash* has a nasty failure mode: it de-risks into a fast drop and is still in cash for the snap-back. In 2020 CPPI sold into the Feb–March crater and missed the recovery — turning buy-and-hold's **+18.3%** year into **-4.8%**. And 2022's slow grind barely tripped the cushion, so it was hardly helped at all.

In [4]:
for yr,bh,c in [(2020,R['crash20_bh'],R['crash20_c']),(2022,R['crash22_bh'],R['crash22_c'])]:
    print(f"{yr}: buy-and-hold {bh:+.1f}%   ->   CPPI {c:+.1f}%")

2020: buy-and-hold +18.3%   ->   CPPI -4.8%
2022: buy-and-hold -18.2%   ->   CPPI -18.7%


## 4. A live synthetic control — the engine is honest

We plant a **bear** world (the floor must visibly hold) and a **calm** null (nothing to protect), and check the machinery reads each correctly. No network.

In [5]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
from cppi import data, strategy as st
bear = st.synthetic_detect(data.synthetic_prices(seed=897, n_days=2500, drift=-0.0004, sigma=0.016)[0])
calm = st.synthetic_detect(data.synthetic_prices(seed=897, n_days=2500, drift=0.0004, sigma=0.006)[0])
print(f"bear world: drawdown protection {bear['dd_protection']:+.3f}, floor breaches {bear['n_breach']}")
print(f"calm null : drawdown protection {calm['dd_protection']:+.3f}  (nothing to protect)")

bear world: drawdown protection +0.538, floor breaches 0
calm null : drawdown protection -0.000  (nothing to protect)


## 5. The honest verdict

- **Signal: Mixed.** The floor-protection claim is **real and mechanical** (maxDD -20.8% vs -55.2%, zero breaches, 2008 cut from −47% to −11%). But the implied *'insurance improves the risk-adjusted outcome'* claim is **false**: CPPI's Sharpe is 0.16 vs 0.54 (CI clear of zero), and its dynamic re-timing *subtracts* value (-1.56%/yr). The floor is real; the free lunch is not.
- **Tradability: Mirage.** As a way to *make money* there is nothing to bank — CPPI trails buy-and-hold gross, before a single basis point of cost. The drawdown cap is cheap and real **as insurance**, but insurance is a cost (2.4% CAGR vs 10.7%), not a paycheck — and an overnight gap past 1/m = 20% would breach even the floor.